# Couzin Zones:
Couzin defines 3 zones, each produces a different response on each of the fish's behaviour.

**Repulsion:** is the primordial. Let's take d = distance, if d<0.5 the fish will move away. Fish avoid collisions.

**Orientation:** if the neighbors are in a comfortable/ideal distance, the fish will calculate the medium of the neighbor's orientation and will move towards the same direction.

**Attraction:** if a fish is in a distance d<5.5, the fish will get close (because it's far neighbor still close to it). otherwise, if the neighbor is in a d>5.5, the fish will ignore it (it's out of its reach).

### For now we are just interesting that the fish recognized its relevant neighbors.

## Fisrt: 
**Attraction:** we detect the fishes that are in a radius of 5.5.

In [1]:
''' Phase 1B: 20 individually fishes moving in continuous space.
Objective: create 20 inidividual Golden Shiners that move through a Continuous Space'''

import numpy as np
import mesa
from mesa.experimental.continuous_space import ContinuousSpaceAgent, ContinuousSpace
from mesa.visualization import SolaraViz, make_space_component
from matplotlib.markers import MarkerStyle

class GoldenShiners(ContinuousSpaceAgent):
    """ Creatin a Golden Shienr """
    def __init__(self, model, space, position=(0, 0), speed=1.0):
        super().__init__(space, model)
        self.position = np.array(position, dtype=float)
        self.speed = speed
     
        # Random initial direction:
        angle = self.model.rng.uniform(0, 2 * np.pi)
        self.direction = np.array([np.cos(angle), np.sin(angle)])

    # def get_neighbors(self):
    #         neighbors = []
    #         for other_fish in self.model.agents: #Now it takes into account ALL fishes, ignoring itself
    #             if other_fish is self:
    #                 continue
    #         #For now we just want the fish to take into account the fishes that are in its reach, menaing under 5.5 of distance:
    #             distance = np.linalg.norm(other_fish.position - self.position) #create vectors from one fish to another and take total distance
    #             if distance <= self.model.attraction_radius:
    #                 neighbors.append(other_fish)
    #         return neighbors

    def get_neighbors(self):
        neighbors = []
        for other_fish in self.model.agents:
            if other_fish is self:
                continue
            distance = np.linalg.norm(other_fish.position - self.position)
            if distance <= self.model.attraction_radius:
                neighbors.append(other_fish)

        return neighbors

    
    def move(self):
        """Move forward in current direction."""
        new_position = self.position + self.direction * self.speed
        self.bounce(new_position)
        self.position = new_position

    #Making sure that the agent changes it's direction when it encounters a wall:
    def bounce(self, position):
        for axis,(minimum, maximum) in enumerate(self.model.bounds):

            if position[axis] < minimum:
                position[axis] = 2 * minimum - position[axis]
                self.direction[axis] *= -1

            elif position[axis] > maximum:
                position[axis] = 2 * maximum - position[axis]
                self.direction[axis] *= -1

                

In [2]:
class GoldenShinersModel(mesa.Model):
    """Phase 1B: 20 fish."""

    def __init__(self, width=100, height=100, speed=1, n_fish= 20, seed=None):
        super().__init__(seed=seed)

        self.bounds = np.array([[0, width],[0, height] ])

        # Fixed Couzin parameters
        self.repulsion_radius = 0.5
        self.orientation_radius = 3.0
        self.attraction_radius = 5.5

        # Create continuous space
        self.space = ContinuousSpace([[0, width], [0, height]], torus=False, random=self.random)

        for _ in range(n_fish):
            # Create n fish
            position = self.rng.random(2) * np.array([width, height]) #chooses a position
            GoldenShiners(self, self.space, position,speed) #creates a fish in that position, in this case 20 fishes

    # def step(self):
    #     """Run one simulation step."""
    #     self.agents.do("move")

    # def step(self):
    #  self.agents.do("move")
    #  for fish in self.agents:
    #     neighbors = fish.get_neighbors()
    #     print(len(neighbors))
    
    def step(self):
        self.agents.do("move")



In [3]:
# Visualization
def agent_draw(agent):
    """Simple agent portrayal with arrow pointing in movement direction."""
    # Calculate angle from direction vector
    angle_rad = np.arctan2(agent.direction[1], agent.direction[0])
    angle_deg = np.degrees(angle_rad)
    
    # Create arrow marker pointing in agent's direction
    marker = MarkerStyle(marker='>')  # Arrow marker
    marker._transform = marker.get_transform().rotate_deg(angle_deg)
    return {"color": "yellow", "size": 15, "marker": marker}

# Set up visualization
# model = GoldenShinersModel()


#To Visualiazite better let's reduce the space pf the fishes temporarily
model = GoldenShinersModel(width=20,height=20,n_fish=20,seed=1)
page = SolaraViz(
    model,
    components=[make_space_component(agent_portrayal=agent_draw)],
    name="Phase 1B: 20 Fishes"
)

page

Cannot show ipywidgets in text

# Trial:
We make sure for the first fish that is being created it can reaches its neughbors

In [8]:
for step in range(20):

    model.step()
    focal_fish = model.agents[0]
    neighbors = focal_fish.get_neighbors()
    print("Step:", step, "|| Neighbors:", len(neighbors)
    )

Step: 0 || Neighbors: 2
Step: 1 || Neighbors: 2
Step: 2 || Neighbors: 2
Step: 3 || Neighbors: 4
Step: 4 || Neighbors: 5
Step: 5 || Neighbors: 5
Step: 6 || Neighbors: 5
Step: 7 || Neighbors: 6
Step: 8 || Neighbors: 6
Step: 9 || Neighbors: 5
Step: 10 || Neighbors: 3
Step: 11 || Neighbors: 3
Step: 12 || Neighbors: 2
Step: 13 || Neighbors: 3
Step: 14 || Neighbors: 4
Step: 15 || Neighbors: 6
Step: 16 || Neighbors: 6
Step: 17 || Neighbors: 7
Step: 18 || Neighbors: 7
Step: 19 || Neighbors: 6
